# Git Rebase: Use Cases and Visual Examples

This notebook explains how `git rebase` works, when to use it, and provides visual examples for different scenarios. It also covers advanced options and best practices.



## What is `git rebase`?

- `git rebase` moves or combines a sequence of commits to a new base commit.
- It is used to maintain a linear project history, update feature branches, or clean up commit history.
- Unlike `git merge`, rebase rewrites commit history.
- **Warning:** Use with caution on shared branches.



## Common Use Cases for `git rebase`

### 1. Update a Feature Branch with Latest Main Branch Changes

#### Scenario:
You have two branches:
- `main`: A -- B -- C
- `feature`: branched from B, with two new commits D and E

#### Before Rebase
```
main:    A -- B -- C
                \
feature:         D -- E (HEAD)
```
- main has commits A, B, C
- feature has A, B, D, E (but not C)

#### Command:
```bash
git checkout feature
git rebase main
```

#### What happens:
1. Temporarily removes D and E from feature
2. Moves feature branch pointer to C (tip of main)
3. Re-applies D and E on top of C as new commits D' and E'

#### After Rebase
```
main:    A -- B -- C
                      \
feature:               D' -- E' (HEAD)
```
- D' and E' are new commits with the same changes as D and E, but now based on C
- Feature branch now includes all changes from main, with a linear history



### 2. Squash Multiple Commits into One (Interactive Rebase)

#### Scenario:
You want to combine several commits into one to clean up your history before merging.

#### Before Rebase
```
A -- B -- C -- D -- E (HEAD)
```
- C, D, and E are separate commits, but you want them as a single commit.

#### Command:
```bash
git rebase -i HEAD~3
# In the editor, mark D and E as 'squash' or 'fixup' to combine with C
```

#### What happens:
1. Git opens an editor listing C, D, E
2. You mark D and E as 'squash' (combine into C)
3. Git creates a new commit C' with all changes from C, D, and E

#### After Rebase
```
A -- B -- C' (HEAD)
```
- C' is a single commit containing the changes from C, D, and E
- History is now cleaner and easier to review



### 3. Rebase a Feature Branch onto a Different Base Branch

#### Scenario:
You want to move your feature branch work from one base to another (e.g., from B to C).

#### Before Rebase
```
main:    A -- B -- C
              \
feature:       D -- E (HEAD)
```
- feature is based on B, but you want it based on C

#### Command:
```bash
git checkout feature
git rebase --onto main B feature
```

#### What happens:
1. Git finds commits on feature after B (D, E)
2. Moves feature branch pointer to C
3. Re-applies D and E on top of C as new commits D', E'

#### After Rebase
```
main:    A -- B -- C
                      \
feature:               D' -- E' (HEAD)
```
- D' and E' are now based on C instead of B
- Useful for moving work from one base to another



### 4. Rebase to Resolve Diverged Branches (Linearize History)

#### Scenario:
Both main and feature have diverged (new commits on both branches). You want to put your feature work on top of the latest main.

#### Before Rebase
```
main:    A -- B -- C -- F
              \
feature:       D -- E (HEAD)
```
- main has new commit F
- feature has D and E, based on B

#### Command:
```bash
git checkout feature
git rebase main
```

#### What happens:
1. Temporarily removes D and E from feature
2. Moves feature branch pointer to F (tip of main)
3. Re-applies D and E on top of F as new commits D', E'

#### After Rebase
```
main:    A -- B -- C -- F
                              \
feature:                       D' -- E' (HEAD)
```
- D' and E' are replayed on top of F, making history linear



## Advanced Options and Best Practices

### 1. `git rebase -i` (Interactive Rebase)
- Lets you reorder, squash, edit, or drop commits.
- Example: `git rebase -i HEAD~3`

### 2. `git rebase --skip`
- Skip the current patch if there is a conflict you don't want to resolve.

### 3. `git rebase --continue`
- Continue the rebase after resolving conflicts.

### 4. `git rebase --abort`
- Abort the rebase and return to the original branch state.

### 5. `git pull --rebase`
- Update your branch with remote changes using rebase instead of merge.

### 6. Never rebase public/shared branches
- Only rebase local or private branches to avoid rewriting shared history.



## Important Rebase Options and Types (with Examples)

Below are the most useful rebase options, what they do, and examples for each.



### 1. `--ff` (Fast-Forward)

- **What it does:** Moves the branch pointer forward if possible, no new commits created.
- **When to use:** When your branch is directly ahead of the base branch.

**Example:**
```
main:    A -- B -- C (HEAD)
feature:           \-- D -- E (HEAD)
# If feature is already up to date with main, this just moves the pointer.

git checkout feature
git rebase --ff main
# Result: No new commits, feature pointer moves to main if possible.
```



### 2. `--no-ff` (No Fast-Forward)

- **What it does:** Forces Git to create a new commit even if a fast-forward is possible.
- **When to use:** To always record a rebase event in history.

**Example:**
```
main:    A -- B -- C (HEAD)
feature:           \-- D -- E (HEAD)

git checkout feature
git rebase --no-ff main

# After:
main:    A -- B -- C -- M (HEAD)
feature:                /
                      D -- E
# M is a new commit created by the rebase, even if a fast-forward was possible.
```



### 3. `-i` / `--interactive` (Interactive Rebase)

- **What it does:** Lets you edit, squash, reorder, or drop commits during the rebase.
- **When to use:** To clean up or rewrite commit history before merging.

**Example:**
```
A -- B -- C -- D -- E (HEAD)

git rebase -i HEAD~3
# In the editor, you can squash, reorder, or drop C, D, E.
# Result: History is rewritten as you choose.
```



### 4. `--preserve-merges` (deprecated) and `--rebase-merges`

- **What it does:** Tries to keep or recreate merge commits during the rebase.
- **When to use:** When you want to keep the logical structure of merges during a rebase.

**Example:**
```
A -- B -- C -- M
           \   /
            D--E

git rebase --rebase-merges main
# Result: Merge commits are recreated in the rebased history.
```
- `--preserve-merges` is deprecated; use `--rebase-merges` instead.



### 5. `--autosquash`

- **What it does:** Automatically squashes or fixes up commits marked with `fixup!` or `squash!` in their message during interactive rebase.
- **When to use:** To automate squashing/fixing up commits during interactive rebase.

**Example:**
```
A -- B -- C -- fixup! B -- squash! C (HEAD)

git rebase -i --autosquash HEAD~4
# Result: Commits with 'fixup!' or 'squash!' are automatically squashed into their targets.
```



### 6. `--skip`, `--continue`, `--abort`

- **What they do:**
  - `--skip`: Skip the current patch if there’s a conflict you don’t want to resolve.
  - `--continue`: Continue after resolving conflicts.
  - `--abort`: Abort the rebase and return to the original state.

**Example:**
```bash
git rebase main
# If there is a conflict:
# (Resolve the conflict)
git rebase --continue
# Or, to skip the commit:
git rebase --skip
# Or, to abort the rebase:
git rebase --abort
```

